# PetPulse: effect size and power analysis

**Claim.** Petting a dog for 5 minutes lowers heart rate more effectively than listening to a guided meditation.

This notebook designs a future randomized experiment. The class prompt does not provide participant-level data, so there is no outcome dataset to analyze yet; the deliverable is the hypothesis-driven design and required sample size.

## 1. Hypotheses and design

Randomly assign adults to either 5 minutes of dog petting (treatment) or 5 minutes of guided meditation (active control). Measure resting heart rate immediately before and after the activity with the same validated monitor. The primary outcome is the change score, `post − pre`, in beats per minute (bpm). Lower values indicate a larger reduction.

- **Null hypothesis (H₀):** the mean change is the same in both groups, μ_dog − μ_meditation = 0.
- **Alternative hypothesis (H₁):** dog petting produces a larger reduction, μ_dog − μ_meditation < 0.

The primary test is a one-sided independent-samples t-test on change scores (Welch's version in the final analysis). Randomization supports a causal interpretation; the active control helps separate dog contact from simply sitting quietly. A two-sided sensitivity analysis is also reported because the literature is heterogeneous.

## 2. Literature-based effect size

The most directly relevant quantitative estimate I found is the meta-analysis by Phillips et al. (2026), which reported a pooled standardized effect of dog presence on heart-rate reactivity of Hedges' g = −0.12 (31 effects; 2,027 participants). This is a **small** effect and is conservative for planning. It is not an exact match: the review studied dog presence during experimental stressors, while PetPulse specifies petting versus meditation and a 5-minute session. I therefore use |d| = 0.12 as a transparent planning proxy, not as proof that the claim is true.

A 2025 systematic review of brief dog-assisted interventions also concluded that heart-rate findings were mixed, supporting the decision to use a conservative effect and to pre-specify the outcome and protocol.

**Sources**
1. Phillips, J. et al. (2026). *The Effect of a Dog on Acute Stress Reactivity During Experimental Stressors: A Systematic Review and Meta-Analysis*. Anthrozoös. DOI: https://doi.org/10.1080/08927936.2026.2621532
2. Faucher, C. et al. (2025). *Effect of brief dog-assisted interventions on psychobiological indicators of stress: a systematic review*. Anxiety, Stress, & Coping. DOI: https://doi.org/10.1080/10615806.2025.2505902

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import nct, t

ALPHA = 0.05
TARGET_POWER = 0.80
EFFECT_SIZE = 0.12  # absolute standardized mean difference, from Phillips et al. (2026)

def one_sided_power(n_per_group, d=EFFECT_SIZE, alpha=ALPHA):
    df = 2 * n_per_group - 2
    noncentrality = d * np.sqrt(n_per_group / 2)
    critical_value = t.ppf(1 - alpha, df)
    return 1 - nct.cdf(critical_value, df, noncentrality)

n_per_group = next(n for n in range(2, 10000) if one_sided_power(n) >= TARGET_POWER)
total_n = 2 * n_per_group
print(f'Required analyzable sample: {n_per_group} per group ({total_n} total)')
print(f'One-sided power at d={EFFECT_SIZE:.2f}: {one_sided_power(n_per_group):.3f}')

The calculation targets α = 0.05 and 80% power for a one-sided test with equal allocation. The result is **860 analyzable participants per group (1,720 total)**. This is large because the best available heart-rate estimate is small. If the study team expects attrition, inflate recruitment: with 15% attrition, recruit approximately 1,012 per group (2,024 total).

In [ ]:
# Sensitivity analysis: how the required sample changes if the true effect differs.
def required_n(d, target_power=TARGET_POWER):
    return next(n for n in range(2, 10000) if one_sided_power(n, d=d) >= target_power)

sensitivity = pd.DataFrame({
    'standardized_effect_d': [0.12, 0.20, 0.30, 0.50],
    'n_per_group': [required_n(d) for d in [0.12, 0.20, 0.30, 0.50]],
})
sensitivity['total_analyzable_n'] = 2 * sensitivity['n_per_group']
sensitivity['total_recruitment_n_at_15pct_attrition'] = np.ceil(sensitivity['total_analyzable_n'] / 0.85).astype(int)
sensitivity

## 3. Analysis plan and rationale

1. Randomize participants and analyze according to assignment (intention-to-treat).
2. Compute each participant's heart-rate change as post-session bpm minus pre-session bpm.
3. Compare mean change between groups with Welch's independent t-test, using the pre-registered one-sided alternative that dog petting yields a lower change. Report the mean difference, 95% confidence interval, p-value, and standardized effect size.
4. Check the distribution of change scores and influential observations. If assumptions are badly violated, report a robust/permutation sensitivity analysis, but do not change the primary test after seeing results.
5. Record session duration, monitor model, baseline heart rate, medication/caffeine, dog familiarity, and protocol deviations. These variables improve measurement quality and permit a pre-specified adjusted sensitivity model; they do not replace randomization.

The claim says 'lowers more effectively,' so a directional alternative is justified, but it must be declared before seeing data. Because the literature shows mixed heart-rate findings, the study should not be described as confirmatory beyond this specific protocol, and the effect-size assumption should be revisited if a pilot or more directly matched study becomes available.

## Conclusion

For the PetPulse claim, the proposed experiment uses an active control and a continuous change-score outcome. Using the conservative literature proxy d = 0.12, α = 0.05, and 80% power requires 1,720 analyzable participants, or about 2,024 recruits after allowing for 15% attrition. The size is intentionally driven by published evidence rather than by choosing a convenient effect.